# Clean OSM SLC restaurants

| File | Role |
|---|---|
| `restaurants.csv` | Overpass snapshot of SLC dining POIs (`amenity` / `shop` / `cuisine`, address, contacts, lat/lon) |
| `overpass_raw.json` | raw Overpass response (optional; not loaded here) |

Join spine: **`osm_type` + `osm_id`** (or `osm_url`).

## Import libraries, define paths, and load dataframes

In [1]:
from pathlib import Path

import pandas as pd

CURR_DIR = Path(".").resolve()
RESTAURANTS_CSV = CURR_DIR / "restaurants.csv"

# OSM tags that classify the place (vs name / address / contacts).
CATEGORY_COLS = ["amenity", "shop", "cuisine"]

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 40)

df = pd.read_csv(RESTAURANTS_CSV)

pd.DataFrame(
    [
        {
            "file": RESTAURANTS_CSV.name,
            "size_mb": round(RESTAURANTS_CSV.stat().st_size / 1e6, 2),
            "exists": RESTAURANTS_CSV.is_file(),
            "rows": len(df),
            "cols": df.shape[1],
        }
    ]
)

,file,size_mb,exists,rows,cols
0,restaurants.csv,0.71,True,983,30


## Preview data types

In [2]:
df.head()

,osm_type,osm_id,name,amenity,shop,cuisine,brand,operator,addr_housenumber,addr_street,...,delivery,outdoor_seating,wheelchair,lat,lon,wikidata,wikipedia,osm_url,tags_json,collected_at
0,node,356683072,Café 140B,cafe,NaN,NaN,NaN,NaN,140,North B Street,...,NaN,yes,NaN,40.772602,-111.881385,NaN,NaN,https://www.openstreetmap.org/node/356683072,"{""addr:city"": ""Salt Lake City"", ""addr:housenumber"": ""140"", ""addr:postcode"": ""84103"", ""addr:state...",2026-08-10T08:57:15.983839+00:00
1,node,369114019,Café Zupas,fast_food,NaN,sandwich;salad;soup,Café Zupas,NaN,NaN,NaN,...,NaN,NaN,yes,40.722240,-111.856832,NaN,NaN,https://www.openstreetmap.org/node/369114019,"{""amenity"": ""fast_food"", ""brand"": ""Café Zupas"", ""brand:wikidata"": ""Q123687995"", ""cuisine"": ""sand...",2026-08-10T08:57:15.983839+00:00
2,node,419857860,Himalayan Kitchen,restaurant,NaN,indian,NaN,NaN,360,South State Street,...,NaN,NaN,NaN,40.761262,-111.888525,NaN,NaN,https://www.openstreetmap.org/node/419857860,"{""addr:city"": ""Salt Lake City"", ""addr:housenumber"": ""360"", ""addr:postcode"": ""84111"", ""addr:state...",2026-08-10T08:57:15.983839+00:00
3,node,489192578,All Chay,restaurant,NaN,vietnamese,NaN,NaN,1264,West 500 North,...,no,no,NaN,40.780540,-111.927431,NaN,NaN,https://www.openstreetmap.org/node/489192578,"{""addr:city"": ""Salt Lake City"", ""addr:housenumber"": ""1264"", ""addr:postcode"": ""84116"", ""addr:stat...",2026-08-10T08:57:15.983839+00:00
4,node,730963653,Feldman's Deli,restaurant,NaN,jewish,NaN,NaN,2005,East 2700 South,...,NaN,NaN,NaN,40.712913,-111.833673,NaN,NaN,https://www.openstreetmap.org/node/730963653,"{""addr:city"": ""Salt Lake City"", ""addr:housenumber"": ""2005"", ""addr:postcode"": ""84106"", ""addr:stat...",2026-08-10T08:57:15.983839+00:00


In [3]:
dtypes = (
    df.dtypes.astype(str)
    .rename("dtype")
    .to_frame()
    .assign(non_null=df.notna().sum(), nulls=df.isna().sum())
)
dtypes

,dtype,non_null,nulls
osm_type,str,983,0
osm_id,int64,983,0
name,str,975,8
amenity,str,928,55
shop,str,61,922
cuisine,str,668,315
brand,str,195,788
operator,str,16,967
addr_housenumber,str,742,241
addr_street,str,743,240


## Unique values per category column

`amenity` / `shop` are mutually exclusive in the collector; `cuisine` can be multi-valued (`;` separated).

In [4]:
pd.DataFrame(
    [
        {
            "column": col,
            # dropna: empty OSM tags are missing, not a real category value
            "n_unique": df[col].nunique(dropna=True),
            "n_non_null": int(df[col].notna().sum()),
            "n_null": int(df[col].isna().sum()),
        }
        for col in CATEGORY_COLS
    ]
)

,column,n_unique,n_non_null,n_null
0,amenity,8,928,55
1,shop,9,61,922
2,cuisine,147,668,315


In [5]:
from IPython.display import display

for col in CATEGORY_COLS:
    print(f"\n=== {col} ({df[col].nunique(dropna=True)} unique) ===")
    display(df[col].value_counts(dropna=False))


=== amenity (8 unique) ===


amenity
restaurant    471
fast_food     170
cafe          147
bar            93
NaN            55
pub            24
ice_cream      16
food_court      6
biergarten      1
Name: count, dtype: int64


=== shop (9 unique) ===


shop
NaN              922
bakery            31
pastry            12
confectionery      7
deli               4
tea                3
ice_cream          1
coffee             1
donut              1
gift               1
Name: count, dtype: int64


=== cuisine (147 unique) ===


cuisine
NaN                                             315
coffee_shop                                      76
mexican                                          70
burger                                           58
pizza                                            47
                                               ... 
tea;wine;coffee_shop                              1
scandinavian;american;breakfast;brunch;lunch      1
steak;fish                                        1
american;burger;pasta;salad                       1
french;italian                                    1
Name: count, Length: 148, dtype: int64